# SFT UDS Pipeline — End-to-End Test (`sft_final/` flat layout)

Runs the whole project in order: **build → batching → shard prep → training + benchmark**.

> Assumes the corrected `sft_final/` contents (all headers present, flat includes).
> GPU recommended for training; everything else runs on CPU.

## 0 · Get the code & locate it
Clones the public repo, then finds the folder that contains `bindings.cpp`.

In [ ]:
import os, sys, glob, subprocess

if not glob.glob("**/bindings.cpp", recursive=True) and not os.path.isdir("/content/sft"):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/SniAssia/sft.git","/content/sft"], check=True)

def find_code_dir():
    roots = ["/content", os.getcwd(), "."]
    for start in roots:
        start = os.path.abspath(start)
        if not os.path.isdir(start): continue
        for root, dirs, files in os.walk(start):
            dirs[:] = [d for d in dirs if d not in (".git","__pycache__","build",".ipynb_checkpoints")]
            if "bindings.cpp" in files:
                return root
    return None

CODE = find_code_dir()
if CODE is None:
    print("Contents of /content:"); print(*sorted(glob.glob("/content/**", recursive=True))[:40], sep="\n")
    raise RuntimeError("bindings.cpp not found — clone/upload the repo first.")
sys.path.insert(0, CODE)
print("code dir:", CODE)
present = sorted(os.path.basename(p) for p in glob.glob(os.path.join(CODE,"*.hpp")))
print("headers :", present)
needed = {"shard_format","sample","length_queues","shard_streamer","distributed",
          "timer","shard_reader","batch_scheduler","collator","prefetcher","data_pipeline"}
missing = sorted(needed - {p[:-4] for p in present})
if missing:
    raise RuntimeError(f"repo is missing headers: {missing} — commit the corrected sft_final/ bundle")
print("all headers present ✅")

In [ ]:
# deps (torch preinstalled on Colab; add transformers)
import importlib
def have(m):
    try: importlib.import_module(m); return True
    except Exception: return False
need=[m for m in ["torch","transformers"] if not have(m)]
if need: subprocess.run([sys.executable,"-m","pip","install","-q",*need], check=True)
import torch; print("torch",torch.__version__,"| CUDA:",torch.cuda.is_available())

## 1 · Build the C++ extension

In [ ]:
b = subprocess.run([sys.executable,"setup.py","build_ext","--inplace"],
                   cwd=CODE, capture_output=True, text=True)
print(b.stdout[-1200:])
if b.returncode!=0:
    print("STDERR:\n", b.stderr[-2500:]); raise RuntimeError("build failed")
import uds_loader
print("uds_loader OK:", [x for x in dir(uds_loader) if not x.startswith("_")])

## 2 · Batching test (model-free)
Whole C++ data path with synthetic ids — no downloads. Checks shapes, masking, padding, Option-B window, and **batch-formation timing**.

In [ ]:
import random, time, prepare_shards as ps
SHARD_DIR=os.path.join(CODE,"_shards_synth"); MAXLEN=2048
cfg=ps.Config(out_dir=SHARD_DIR,max_seq_length=MAXLEN,shard_size=500,
              band_short_max=512,band_medium_max=1536)
w=ps.ShardWriter(cfg); random.seed(0)
for _ in range(1500):
    r=random.random()
    t=(random.randint(20,500) if r<.4 else random.randint(512,1500) if r<.7
       else random.randint(1536,2048) if r<.9 else random.randint(2100,4000))
    pl,cl=max(1,t//5),t//5; rl=t-pl-cl; mk=lambda n:[random.randint(1,84000) for _ in range(n)]
    c,isk,bd=ps.decide_case(pl,cl,rl,MAXLEN); w.add(ps.TokSample(mk(pl),mk(cl),mk(rl),isk,c,bd))
w.flush(); print("bands S/M/L/C:",w.band_counts)

In [ ]:
pc=uds_loader.PipelineConfig()
pc.shards=sorted(glob.glob(os.path.join(SHARD_DIR,"shard_*.bin")))
pc.B=8; pc.homogeneous=True; pc.chunked_rate=0.25; pc.pad_id=0
pc.option_b_window=MAXLEN; pc.pad_to_multiple=8; pc.num_epochs=2
pc.prefetch_workers=3; pc.ring_capacity=4
pipe=uds_loader.DataPipeline(pc); pipe.start()
fit=chunked=seen=0
for i in range(40):
    pool=pipe.next_pool()
    if pool is None: break
    ii,am,lb=pool.input_ids,pool.attention_mask,pool.labels
    assert ii.shape==am.shape==lb.shape and ii.dtype==torch.int64
    assert (lb[am==0]==-100).all() and ii.shape[1]%8==0
    if pool.is_chunked: chunked+=1; assert ii.shape[1]==MAXLEN
    else: fit+=1
    seen+=len(pool)
    if i<6: print(f"pool {i}: shape={tuple(ii.shape)} band={pool.band} chunked={pool.is_chunked} form={pool.formation_seconds*1e3:.2f}ms")
pipe.stop()
print(f"\nfit={fit} chunked={chunked} samples={seen} | C++ formation {pipe.formation_mean_ms():.3f}ms  stall {pipe.stall_mean_ms():.3f}ms")
print("BATCHING PASSED ✅")

## 3 · Real shards (Jais tokenizer)
Tiny Alpaca+chat dataset → offline tokenizer stage. Small `max_seq_length` for a fast CPU demo.

In [ ]:
import json
DATA=os.path.join(CODE,"_data"); os.makedirs(DATA,exist_ok=True)
alp=[{"instruction":f"Add {a} and {b}.","input":"","output":f"The sum is {a+b}."} for a in range(1,40) for b in range(1,6)]
cht=[{"conversations":[{"from":"human","value":f"Say hi to {n}."},{"from":"gpt","value":f"Hello, {n}!"}]} for n in ["Sara","Omar","Lina","Ali","Nour","Yusuf","Maya","Sami"]]
open(f"{DATA}/alpaca.jsonl","w").write("\n".join(map(json.dumps,alp)))
open(f"{DATA}/chat.jsonl","w").write("\n".join(map(json.dumps,cht)))
json.dump([{"name":"alpaca","path":f"{DATA}/alpaca.jsonl","format":"alpaca","source":"jsonl"},
           {"name":"chat","path":f"{DATA}/chat.jsonl","format":"chat","source":"jsonl"}],
          open(f"{DATA}/datasets.json","w"))
print("wrote", len(alp),"alpaca +",len(cht),"chat")

In [ ]:
REAL=os.path.join(CODE,"_shards_real")
r=subprocess.run([sys.executable,"prepare_shards.py","--config",f"{DATA}/datasets.json",
    "--out",REAL,"--tokenizer","inceptionai/jais-family-590m",
    "--max-seq-length","256","--shard-size","128","--workers","2"],
    cwd=CODE, capture_output=True, text=True)
print(r.stdout[-1200:])
if r.returncode!=0: print("STDERR:\n",r.stderr[-2500:]); raise RuntimeError("shard prep failed")
meta=json.load(open(os.path.join(REAL,"meta.json")))
print("meta:",{k:meta[k] for k in ["vocab_size","pad_id","max_seq_length","num_samples","band_counts"]})

## 4 · Training + benchmark
UDS-selected SFT for a few steps. Reports **batch formation / training / total** + prefetch overlap.

In [ ]:
from transformers import AutoModelForCausalLM
from uds import UDSConfig, UDSSelector
from benchmark import Benchmark
device="cuda" if torch.cuda.is_available() else "cpu"
STEPS,B,K,WARMUP=12,8,4,3
model=AutoModelForCausalLM.from_pretrained("inceptionai/jais-family-590m",
        trust_remote_code=True, torch_dtype=torch.float32).to(device); model.train()
opt=torch.optim.AdamW(model.parameters(),lr=1e-5)
sel=UDSSelector(int(meta["vocab_size"]),
        UDSConfig(K=K,alpha=1.0,svd_proj_dim=128,fp_dim1=64,fp_dim2=8,
                  start_sampling_step=WARMUP,device=device))

In [ ]:
pc=uds_loader.PipelineConfig(); pc.shards=sorted(glob.glob(os.path.join(REAL,"shard_*.bin")))
pc.B=B; pc.homogeneous=True; pc.chunked_rate=0.0; pc.pad_id=int(meta["pad_id"])
pc.option_b_window=int(meta["max_seq_length"]); pc.pad_to_multiple=8
pc.num_epochs=-1; pc.prefetch_workers=2; pc.ring_capacity=3
pipe=uds_loader.DataPipeline(pc); pipe.start()
bench=Benchmark(sync=lambda: torch.cuda.synchronize() if torch.cuda.is_available() else None); bench.start()
for step in range(STEPS):
    with bench.phase("batch_formation"): pool=pipe.next_pool()
    if pool is None: break
    ii=pool.input_ids.to(device); am=pool.attention_mask.to(device); lb=pool.labels.to(device)
    if step<WARMUP: t_ii,t_am,t_lb=ii,am,lb
    else:
        with bench.phase("uds_scoring",gpu=True):
            with torch.no_grad(): logits=model(input_ids=ii,attention_mask=am).logits
            s,z,si,se=sel.score(logits,am); idx=sel.select(s); sel.commit(z[idx])
        t_ii,t_am,t_lb=ii[idx],am[idx],lb[idx]
    with bench.phase("training",gpu=True):
        opt.zero_grad(set_to_none=True)
        out=model(input_ids=t_ii,attention_mask=t_am,labels=t_lb); out.loss.backward(); opt.step()
    bench.add_samples(t_ii.shape[0],tokens=int(t_am.sum().item()))
    print(f"step {step:2d} | pool={len(pool)} train={t_ii.shape[0]} loss={out.loss.item():.4f}")
bench.stop(); pipe.stop()

In [ ]:
cpp={"formation_total_s":pipe.formation_total_s(),"formation_mean_ms":pipe.formation_mean_ms(),
     "stall_total_s":pipe.stall_total_s(),"stall_mean_ms":pipe.stall_mean_ms()}
print(bench.report(cpp)); bench.save(os.path.join(CODE,"benchmark.json"),cpp)

## 5 · Multi-GPU (DDP)
```bash
torchrun --nproc_per_node=8 train.py --shards ./_shards_real \
    --model inceptionai/jais-family-590m --steps 200 --B 64 --K 32 --warmup 20
```